# UCI Cardiovascular Risk Machine Learning Project

**Project Goal:**

In this notebook I analyze the UCI Heart Disease dataset (preprocessed) to predict heart disease risk using selected features.

**Objectives:**

1. Analyze and clean the data
2. Perform Exploratory Data Analysis (EDA)
3. Conduct feature engineering and risk stratification
4. Build an interpretable logistic regression model
5. Interpret model results and provide clinical insights
6. Summarize key findings, hypotheses, and next steps

## Analyzing and Cleaning Data

**Project Goal:**

In this section I analyze and clean the data to prepare it for Exploratory Data Analysis (EDA). I will also create the variable map, load the data, and define the target variable.

**Objectives:**

- Define the **target variable**
- Create the **variable map**
- Load data
- Name columns
- Inspect data
- Clean data

### Target Variable

**Project Goal:**

This project aims to predict heart disease risk using available clinical features.

**Definition:**

- **Target Variable:** `target` — Heart disease (binary: 0 = no disease, 1 = disease)
- **Predictor Variables:** Clinical features used to predict heart disease risk:
  - `age`
  - `sex`
  - `cp`
  - `trestbps`
  - `chol`
  - `fbs`
  - `restecg`
  - `thalach`
  - `exang`
  - `oldpeak`
  - `slope`
  - `ca`
  - `thal`

### Dataset and Variable Map

The Cleveland UCI Heart Disease dataset contains 14 columns.

**Variable Map**

### Required imports

- `pandas`
- `numpy`
- `matplotlib.pyplot`
- `seaborn`
- `sklearn` (used functions below):
  - `LogisticRegression`
  - `train_test_split`
  - `classification_report`
  - `confusion_matrix`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

-------------------------------------------------------------

### Load and title the dataset

I am loading the preprocessed Cleveland UCI Heart Disease dataset. This dataset does not contain column headers, so I will manually insert them for readability and later use.

The dataset is stored in the pandas DataFrame `df`.

In [ ]:

columns = ["age", "sex", "cp", "trestbps", "chol", "fbs",
           "restecg", "thalach", "exang", "oldpeak", "slope",
           "ca", "thal", "target"]

df = pd.read_csv("heart_disease.csv", header=None, names=columns)

-------------------------------------------------------------

### Checking the dataset

I am checking the dataset to ensure everything is in order.

The data is successfully loaded and stored in **df**.

All column headers have been inserted correctly.

In [ ]:
print(df.head())

-------------------------------------------------------------

### Taking stock of the data

It's time to take stock of the dataset:

First, we check the dataset size. We have:
- **14** columns
- **303** rows (indexed 0 through 302)

### Data Cleaning and Preprocessing

For data processing, we need no missing values and numeric types where appropriate.

We use `df.info()` to inspect types and missingness.

Although `df.info()` initially shows no nulls, two columns are typed as `object` because the preprocessed dataset used `'?'` to indicate missing values.

Therefore, the next step is to replace `'?'` with proper null values so we can accurately assess and handle missing data.

In [ ]:
print(df.info())

-------------------------------------------------------------

### Replace `?` with proper nulls

The next step is to replace `?` with `pd.NA` so pandas treats them as missing values. We do this with `df.replace()`.

- Replace: `'?'`
- With: `pd.NA`


In [ ]:
df.replace('?', pd.NA, inplace=True)

In [ ]:
df.info()

-------------------------------------------------------------

#### Confirming the replacement

The cell below confirms the replacement was successful.

We expect to see missing values in:

- **`ca`**: 4 nulls
- **`thal`**: 2 nulls

These columns can then be converted to numeric types.

In [ ]:
print(df.isnull().sum())

-------------------------------------------------------------

### Drop rows with missing values

Now we drop rows with missing values using `df.dropna()` and verify that no features contain nulls afterward.

In [ ]:
df.dropna(inplace=True) 
print(df.isnull().sum())

-------------------------------------------------------------

### Converting features to numeric

When we first inspect the `ca` and `thal` columns, they appear as `object` because their values are stored as strings.

We convert these string values to numeric types (floats/ints) in the cell below.

After conversion, the previous type issues should be resolved.

In [ ]:
print(df['ca'].apply(repr).unique())
print(df['ca'].map(type).value_counts())

print(df['thal'].apply(repr).unique())
print(df['thal'].map(type).value_counts())

In [ ]:
df['ca'] = pd.to_numeric(df['ca'])
df['thal'] = pd.to_numeric(df['thal'])

------------------------------------------------------------------------------------------------------

### Validating numeric ranges

Next, we ensure that numeric values fall within plausible ranges:

- Age: not negative and not greater than 125
- Blood pressure: not negative
- Categorical variables: values within allowed ranges

------------------------------------------------------------------------------------------------------

#### Confirming age falls within a plausible range

I use `describe()` to check the **min** and **max** values.

- **min**: 29
- **max**: 77

This indicates ages fall within a plausible range.

(There are other checks possible, but `describe()` provides a quick sanity check.)

In [ ]:
print(df['age'].describe())

-------------------------------------------------------------

#### Confirming blood pressure values are non-negative

The following code checks that all blood pressure values are valid.

If no negative values are found, the check prints "All blood pressure values are valid."

In [ ]:
for value in df['trestbps']:
    if value < 0:
        print("Negative blood pressure values found.")
        break
else:
    print("All blood pressure values are valid.")

-------------------------------------------------------------

#### Confirming categorical values fall within expected ranges

Our categorical columns and expected ranges:

| Column | Range | Values Within Range |
|--------|--------------|-------------|
| sex | (0 = female, 1 = male) | yes |
| cp | (chest pain type: 1–4) | yes |
| fbs | (fasting blood sugar > 120 mg/dl: 0 = false, 1 = true) | yes |
| restecg | (0 = normal, 1 = ST-T abnormality, 2 = LVH) | yes |
| exang | (exercise-induced angina: 0 = no, 1 = yes) | yes |
| slope | (1 = upsloping, 2 = flat, 3 = downsloping) | yes |
| thal | (3 = normal, 6 = fixed defect, 7 = reversible defect) | review |
| target | (0 = no heart disease, 1 = yes) | yes |

We will check each column programmatically and address any values outside the expected ranges.

In [ ]:
# Validating 'sex' column

for value in df['sex']:
    if value not in [0, 1]:
        print("Out of Range")
        break
else:
    print("All sex values are valid.")

In [ ]:
# Validating 'cp' column

for value in df['cp']:
    if value not in [1, 2, 3, 4]:
        print("Out of Range")
        break
else:
    print("All cp values are valid.")

In [ ]:
# Validating 'fbs' column

for value in df['fbs']:
    if value not in [0, 1]:
        print("Out of Range")
        break
else:
    print("All fbs values are valid.")

In [ ]:
# Validating 'restecg' column

for value in df['restecg']:
    if value not in [0, 1, 2]:
        print("Out of Range")
        break
else:
    print("All restecg values are valid.")

In [ ]:
# Validating 'exang' column

for value in df['exang']:
    if value not in [0, 1]:
        print("Out of Range")
        break
else:
    print("All exang values are valid.")

In [ ]:
# Validating 'slope' column

for value in df['slope']:
    if value not in [1, 2, 3]:
        print("Out of Range")
        break
else:
    print("All slope values are valid.")

In [ ]:
# Validating 'thal' column

for value in df['thal']:
    if value not in [3, 6, 7]:
        print("Out of Range")
        break
else:
    print("All thal values are valid.")

In [ ]:
# Validating 'target' column

for value in df['target']:
    if value not in [0, 1]:
        print("Out of Range")
        break
else:
    print("All target values are valid.")

-------------------------------------------------------------

### Putting the target into range

The original dataset encodes disease severity with values 0–4. For simplicity, we convert this to a binary target.

| Value | Key |
|---|---|
| 0 | No presence of heart disease |
| 1 | Mild heart disease |
| 2 | Moderate heart disease |
| 3 | Severe heart disease |
| 4 | Very severe heart disease |

For modeling we map values 1–4 to 1 (presence of heart disease), producing a binary target (`0`, `1`). After running the code below, all target values will be binary and in range.

In [ ]:
df['target'].replace([1, 2, 3, 4], 1, inplace=True)

In [ ]:
df['target'].apply(repr).unique()

-------------------------------------------------------------

### Number of rows dropped

Number of rows dropped:

- 6 rows due to missing values

-------------------------------------------------------------

### Ready to move on

We have now:

- Established the target variable
- Created the variable map
- Loaded the data
- Cleaned the data

We are now ready for **Exploratory Data Analysis (EDA)**

-------------------------------------------------------------

## Perform Exploratory Data Analysis (EDA)

**Project Goal:**

In this section I will explore the cleaned dataset to understand distributions, patterns, and relationships between variables. The goal is to identify trends and potential associations between clinical features and the presence of heart disease before modeling.

**Objectives:**

- Analyze distributions of **numeric variables**
- Compare numeric features by **heart disease status**
- Examine **categorical variables** in relation to the target
- Identify patterns and trends relevant to cardiovascular risk
- Create visualizations to support exploratory findings
- Interpret results to inform modeling decisions

-------------------------------------------------------------

### Performing numeric EDA

In this part we perform EDA on the numeric variables.

Goals:

- Compute summary statistics for numeric variables (mean, median, min, max, standard deviation, and range)
- Compare numeric features by target (heart disease vs. no heart disease):
  - Age
  - Cholesterol (`chol`)
  - Resting blood pressure (`trestbps`)
  - Maximum heart rate (`thalach`)
- Check distributions and spread
- Interpret observed differences

-------------------------------------------------------------

### Statistical summary

Here we determine mean, median, min, max, standard deviation, range, and the 25% and 75% percentiles.

This is for the entire dataset.

We show the table:

| Statistics | age | chol | trestbps | thalach | oldpeak |
|---|---:|---:|---:|---:|---:|
| Mean | 54.542 | 247.350 | 131.694 | 149.599 | 1.055 |
| 25% | 48 | 211 | 120 | 133 | 0 |
| Median | 56 | 243 | 130 | 153 | 0.8 |
| 75% | 61 | 276 | 140 | 166 | 1.6 |
| Min | 29 | 126 | 94 | 71 | 0 |
| Max | 77 | 564 | 200 | 202 | 6.2 |
| Range | 48 | 438 | 106 | 131 | 6.2 |
| Standard deviation | 9.050 | 51.998 | 17.763 | 22.942 | 1.166 |



In [ ]:
stats_df = df[['age', 'chol', 'trestbps', 'thalach', 'oldpeak']].describe()
# Add a 'range' row (max - min)
stats_df.loc['range'] = stats_df.loc['max'] - stats_df.loc['min']
# Rename the '50%' row to 'median' (rename index explicitly)
stats_df.rename(index={'50%': 'median'}, inplace=True)
# Also keep the standard deviation under a more readable label
stats_df.loc['standard deviation'] = stats_df.loc['std']
# Select and order the rows we want to display (use 'median' not '50%')
out_fields = ['mean','25%','median','75%','min','max','range','standard deviation']
stats_df = stats_df.loc[out_fields]

print(stats_df)

### Statistical summary by target

Now I show statistics for each group:
- Patients without heart disease
- Patients with heart disease

#### Patients without heart disease:

| Statistics | age | chol | trestbps | thalach | oldpeak |
|---|---:|---:|---:|---:|---:|
| Mean | 52.643 | 243.493 | 129.175 | 158.581 | 0.598 |
| 25% | 45 | 209 | 120 | 149 | 0 |
| Median | 52 | 236 | 130 | 161 | 0.2 |
| 75% | 59 | 268 | 140 | 172 | 1.1 |
| Min | 29 | 126 | 94 | 96 | 0 |
| Max | 76 | 564 | 180 | 202 | 4.2 |
| Range | 47 | 438 | 86 | 106 | 4.2 |
| Standard deviation | 9.551 | 53.757 | 16.373 | 19.043 | 0.787 |

#### Patients with heart disease:

| Statistics | age | chol | trestbps | thalach | oldpeak |
|---|---:|---:|---:|---:|---:|
| Mean | 56.759 | 251.854 | 134.635 | 139.109 | 1.589 |
| 25% | 53 | 218 | 120 | 125 | 0.6 |
| Median | 58 | 284 | 130 | 142 | 1.4 |
| 75% | 62 | 284 | 145 | 157 | 2.5 |
| Min | 35 | 131 | 100 | 71 | 0 |
| Max | 77 | 409 | 200 | 195 | 6.2 |
| Range | 42 | 278 | 100 | 124 | 6.2 |
| Standard deviation | 7.899 | 49.679 | 18.896 | 22.710 | 1.305 |

Values for percentiles and range are rounded to the nearest whole number (except for `oldpeak`). Mean and standard deviation are shown to three decimal places.

In [ ]:
numeric_cols = ['age', 'chol', 'trestbps', 'thalach', 'oldpeak']

# Patients without heart disease
stats_target0 = df[df['target']==0][numeric_cols].describe().transpose()
stats_target0['range'] = stats_target0['max'] - stats_target0['min']
stats_target0['standard deviation'] = stats_target0['std']
stats_target0.rename(columns={'50%':'median'}, inplace=True)
stats_target0 = stats_target0[['mean','25%','median','75%','min','max','range','standard deviation']]

# Patients with heart disease
stats_target1 = df[df['target']==1][numeric_cols].describe().transpose()
stats_target1['range'] = stats_target1['max'] - stats_target1['min']
stats_target1['standard deviation'] = stats_target1['std']
stats_target1.rename(columns={'50%':'median'}, inplace=True)
stats_target1 = stats_target1[['mean','25%','median','75%','min','max','range','standard deviation']]

print("Statistics for patients without heart disease (target=0):")
print(stats_target0)

print("\nStatistics for patients with heart disease (target=1):")
print(stats_target1)

### Key findings

#### Overall dataset
- Most patients are aged **48–61 years**, with outliers from 29 to 77.
- **Cholesterol** varies widely (126–564 mg/dL), indicating some extreme cases.
- Resting **blood pressure** is mostly moderate (120–140 mmHg), but some patients have very high values.
- **Maximum heart rate** ranges widely (71–202 bpm), reflecting differences in cardiovascular fitness.
- **Oldpeak** ranges from 0 to 6.2, showing that some patients experience significant ST depression under exercise stress.

#### Patients without heart disease
- Generally **younger**, with 25% under 45.
- **Higher maximum heart rates**, with 75% above 161 bpm, indicating better cardiovascular response.
- Cholesterol and blood pressure are mostly moderate, with some extreme values.
- **Oldpeak** is low (median 0.2), with most patients experiencing minimal ST depression, suggesting good cardiac response to exercise.

#### Patients with heart disease
- **Older**, with 25% above 62 years.
- **Lower maximum heart rates**, with 25% below 125 bpm, indicating reduced cardiac performance.
- **Higher cholesterol** in the upper quartiles, suggesting a link to heart disease.
- Slightly higher resting blood pressure than healthy patients.
- **Oldpeak** is higher (median 1.4), with the upper quartile reaching 2.5, indicating more significant ST depression and likely exercise-induced ischemia.

#### Key comparisons
- Heart disease patients are **older** (~4 years on average).
- **Cholesterol** tends to be higher, especially in the upper quartiles.
- **Resting blood pressure** is slightly higher.
- **Maximum heart rate** is lower, indicating reduced cardiovascular capacity.
- **Oldpeak** is substantially higher, showing worse cardiac response to stress in patients with heart disease.
- Quartile differences are more pronounced than average differences, highlighting the importance of extremes in age, cholesterol, heart rate, and exercise-induced ST depression.

------------------------------------------------------------------------------------------------------

### Interpreting numerical data through graphs

For numeric variables I will provide the following graphs:

- Boxplot
- Violin plot
- Histogram
- Boxen plot
- Seaborn pairplot

I will use these to compare numeric features with the target variable.

### Boxplot interpretation

We will create a boxplot to compare distributions.

Purpose:
- Compare the distribution of a numeric variable across categories
- Show median, quartiles, and outliers
- Identify differences in numeric features between patients with and without heart disease

### Boxplot Age

The boxplot shows that patients with heart disease tend to be older, with a median age around 58 compared to about 52 for those without heart disease. There is also more variability in age among patients without heart disease, while heart disease patients have a narrower age range with a few younger and older outliers.


In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x='target', y='age', data=df)
plt.title("Age Distribution by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Age")

### Boxplot — Cholesterol

The boxplot shows cholesterol levels are slightly higher on average for individuals with heart disease than for those without, though there is considerable overlap. Both groups include high outliers, indicating a few individuals with unusually elevated cholesterol levels.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x='target', y='chol', data=df)
plt.title("Cholesterol Distribution by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Cholesterol")

### Boxplot — Resting Blood Pressure

The boxplot shows individuals with heart disease have a slightly higher median resting blood pressure compared to those without heart disease, although the distributions overlap substantially. There is greater variability and several high outliers among individuals with heart disease, indicating more extreme blood pressure values in this group.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x='target', y='trestbps', data=df)
plt.title("Resting Blood Pressure Distribution by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Resting Blood Pressure")

### Boxplot — Maximum Heart Rate Achieved

The boxplot shows individuals with heart disease generally achieve lower maximum heart rates than those without heart disease. While there is overlap between the groups, the non-disease group has a higher median and wider spread, with several high outliers indicating better cardiovascular capacity.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x='target', y='thalach', data=df)
plt.title("Maximum Heart Rate Achieved by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Maximum Heart Rate Achieved")

### Boxplot — Exercise-induced ST Depression

The boxplot indicates patients with heart disease tend to have higher exercise-induced ST depression than those without heart disease. The median ST depression is higher in the heart disease group, and there is a greater density of observations at elevated values. While both groups overlap, lower ST depression is more common in patients without heart disease. These patterns suggest higher exercise-induced ST depression is associated with increased heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(x='target', y='oldpeak', data=df)
plt.title("Exercise-induced ST depression by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Exercise-induced ST depression")

------------------------------------------------------------------------------------------------------

### Violin plot interpretation

Purpose:

- Similar to a boxplot but also shows the density of the data
- Reveal the shape and spread of the distribution
- Identify peaks (modes) where most observations cluster
- Show median and quartiles
- Highlight symmetry or skewness

### Violin plot — Age

The violin plot shows patients with heart disease tend to be older than those without. The median age is higher in the heart disease group, with a greater density in the 50–65 age range. While both groups span similar ages, younger patients are more prevalent in the non-disease group, suggesting age is associated with heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.violinplot(x='target', y='age', data=df)
plt.title("Age Distribution by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Age")
plt.show()

### Violin plot — Cholesterol

The violin plot shows patients with heart disease tend to have slightly higher cholesterol levels than those without. The median cholesterol is higher in the heart disease group, and many patients cluster between approximately 220 and 270 mg/dL. Both groups exhibit a wide range of values, and a few extreme high values likely reflect true clinical variation. These patterns suggest elevated cholesterol may be associated with heart disease, although there is considerable overlap.

In [ ]:
plt.figure(figsize=(8,6))
sns.violinplot(x='target', y='chol', data=df)
plt.title("Cholesterol Distribution by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Cholesterol (mg/dl)")
plt.show()

### Violin plot — Resting Blood Pressure

The violin plot shows patients with heart disease tend to have slightly higher resting blood pressure than those without heart disease. The median resting blood pressure is higher in the heart disease group, with a higher density around 130–140 mmHg. While both groups span a similar range, lower blood pressures are more common in the non-disease group, suggesting resting blood pressure may be moderately associated with heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.violinplot(x='target', y='trestbps', data=df)
plt.title("Resting Blood Pressure Distribution by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Resting Blood Pressure (mmHg)")
plt.show()

### Violin plot — Maximum Heart Rate

The violin plot shows patients with heart disease tend to have lower maximum heart rates than those without heart disease. The median maximum heart rate is lower in the heart disease group, with a density peak around 140–160 bpm. While both groups overlap, higher maximum heart rates are more prevalent in the non-disease group, suggesting maximum heart rate is associated with heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.violinplot(x='target', y='thalach', data=df)
plt.title("Maximum Heart Rate Distribution by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Maximum Heart Rate (bpm)")
plt.show()

### Violin plot — Exercise-induced ST Depression

The violin plot shows patients with heart disease tend to have higher exercise-induced ST depression compared to those without heart disease. The median ST depression is higher in the heart disease group, and there is a greater density of observations at elevated values. These patterns suggest higher exercise-induced ST depression is associated with increased heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.violinplot(x='target', y='oldpeak', data=df)
plt.title("Exercise-induced ST depression by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Exercise-induced ST depression")
plt.show()

------------------------------------------------------------------------------------------------------

### Histogram interpretation

Purpose:

- Show the frequency distribution of a numeric variable
- Visualize the shape of the distribution
- Compare groups by coloring with a hue
- Identify skewness, peaks, and spread

### Histogram — Age

The histogram shows patients with heart disease tend to be older than those without. The heart disease group peaks around 55–65, while the non-disease group has a wider spread with more younger patients. These patterns suggest age is associated with heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(data=df, x='age', hue='target', kde=True, element='step', palette='Set1')
plt.title("Age Distribution by Heart Disease Status")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

### Histogram — Cholesterol

The histogram shows that cholesterol levels are broadly similar between patients with and without heart disease, but there is slightly greater density of observations in the 200–250 mg/dl range for the heart disease group. Both groups span a wide range of cholesterol values, though extremely high cholesterol is somewhat more common in the heart disease group. These patterns suggest cholesterol may be associated with heart disease, but the overlap indicates it is not a strong distinguishing factor on its own.

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(data=df, x='chol', hue='target', kde=True, element='step', palette='Set1')
plt.title("Cholesterol Distribution by Heart Disease Status")
plt.xlabel("Cholesterol (mg/dl)")
plt.ylabel("Count")
plt.show()

### Histogram Resting Blood Pressure

The histogram shows that patients with heart disease tend to have slightly higher resting blood pressure than those without heart disease. The distribution of blood pressure in the heart disease group peaks around 130–140 mm Hg, while the non-disease group has a more even spread across lower values. These patterns suggest that higher resting blood pressure is moderately associated with heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(data=df, x='trestbps', hue='target', kde=True, element='step', palette='Set1')
plt.title("Resting Blood Pressure Distribution by Heart Disease Status")
plt.xlabel("Resting BP (mm Hg)")
plt.ylabel("Count")
plt.show()

### Histogram — Maximum Heart Rate Achieved

The histogram shows that patients with heart disease tend to have lower maximum heart rates than those without heart disease. The distribution of maximum heart rate in the non-disease group peaks around 160–170 bpm, while the heart disease group peaks around 140–150 bpm. These patterns suggest that lower maximum heart rate is associated with heart disease, though there is overlap between the groups.

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(data=df, x='thalach', hue='target', kde=True, element='step', palette='Set1')
plt.title("Max Heart Rate Distribution by Heart Disease Status")
plt.xlabel("Max Heart Rate (bpm)")
plt.ylabel("Count")
plt.show()

### Histogram Exercise-induced ST depression

The histogram shows that patients with heart disease tend to have higher exercise-induced ST depression compared to those without heart disease. The distribution for the heart disease group peaks at higher oldpeak values, while the non-disease group has more observations at lower values. Both groups span a similar range, but elevated ST depression is more common in the heart disease group. These patterns suggest that higher exercise-induced ST depression is associated with increased risk of heart disease.

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(data=df, x='oldpeak', hue='target', kde=True, element='step', palette='Set1')
plt.title("Exercise-induced ST depression by Heart Disease Status")
plt.xlabel("Exercise-induced ST depression")
plt.ylabel("Count")
plt.show()

------------------------------------------------------------------------------------------------------

### Boxen plot interpretation

Purpose:

- Similar to a boxplot but shows more quantiles.
- Useful for large datasets or variables with many extreme values.
- Provides a detailed view of tails and outliers.
- Especially helpful for variables like cholesterol or blood pressure when distributions have long tails.

### Boxen plot — Age

The boxen plot shows that patients with heart disease tend to be older than those without. The median age is higher in the heart disease group, and the boxes reveal greater density of observations in the 50–65 age range. While both groups span a similar age range, younger patients are more prevalent in the non-disease group. These patterns suggest age is associated with heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxenplot(x='target', y='age', data=df)
plt.title("Age Distribution (Detailed) by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Age")
plt.show()

### Boxen plot — Cholesterol

The boxen plot shows cholesterol levels are broadly similar between patients with and without heart disease. The median cholesterol is slightly higher in the heart disease group, and the boxes indicate a modest density of observations around 220–250 mg/dL. Both groups span a wide range of values; very high cholesterol is somewhat more prevalent in the heart disease group. These patterns suggest cholesterol may be associated with heart disease, but overlap limits its standalone predictive power.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxenplot(x='target', y='chol', data=df)
plt.title("Cholesterol Distribution (Detailed) by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Cholesterol (mg/dl)")
plt.show()

### Boxen plot — Resting Blood Pressure

The boxen plot shows that patients with heart disease tend to have slightly higher resting blood pressure than those without. The median resting blood pressure is higher in the heart disease group, and the boxes show greater density around 130–140 mmHg. While both groups span a similar range, lower blood pressures are more common in the non-disease group. These patterns suggest higher resting blood pressure is moderately associated with heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxenplot(x='target', y='trestbps', data=df)
plt.title("Resting Blood Pressure Distribution (Detailed) by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Resting Blood Pressure (mmHg)")
plt.show()

### Boxen plot — Maximum Heart Rate Achieved

The boxen plot shows that patients with heart disease tend to have lower maximum heart rates than those without. The median maximum heart rate is lower in the heart disease group, and the boxes indicate greater density in the 140–160 bpm range. While both groups overlap, higher maximum heart rates are more prevalent in the non-disease group, suggesting lower maximum heart rate is associated with heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxenplot(x='target', y='thalach', data=df)
plt.title("Maximum Heart Rate Achived Distribution (Detailed) by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Max Heart Rate (bpm)")
plt.show()

### Boxen plot — Exercise-induced ST Depression

The boxen plot shows that patients with heart disease tend to have higher exercise-induced ST depression than those without. The median ST depression is higher in the heart disease group, and the boxes reveal a greater density of elevated values. While both groups span a similar range, lower ST depression is more common in patients without heart disease. These patterns suggest higher exercise-induced ST depression is associated with increased heart disease risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.boxenplot(x='target', y='oldpeak', data=df)
plt.title("Exercise-induced ST depression Distribution (Detailed) by Heart Disease Status")
plt.xlabel("Heart Disease (0=No, 1=Yes)")
plt.ylabel("Exercise-induced ST depression")
plt.show()

### Pairplot interpretation

**Purpose:**

- Create scatterplots for numeric variables against each other.
- Explore relationships between numeric features before modeling.
- Identify potential correlations and feature interactions.
- Observe clustering between target groups (heart disease vs. no heart disease).

This visualization shows how numeric features correlate, separated by heart disease status. Key findings follow.

#### Feature-level observations

**Age**
- Patients with heart disease tend to be older.
- Age is associated with heart disease risk.

**Cholesterol**
- Distributions largely overlap between groups.
- Cholesterol alone is not a strong predictor in this dataset.

**Resting blood pressure**
- Patients with heart disease tend to have slightly higher resting blood pressure.
- Resting blood pressure is moderately associated with heart disease risk.

**Maximum heart rate**
- Patients with heart disease tend to have lower maximum heart rates.
- Maximum heart rate is strongly associated with heart disease risk.

**Exercise-induced ST depression (oldpeak)**
- Patients with heart disease tend to have higher oldpeak values.
- Lower oldpeak values are more common in patients without heart disease.
- Oldpeak shows clear separation between target groups, indicating a strong association with heart disease risk.

#### Correlations between features

**Age vs Maximum Heart Rate**
- Older patients tend to achieve lower maximum heart rates.
- This negative relationship is more pronounced in patients with heart disease.

**Age vs Resting Blood Pressure**
- Older patients often have slightly higher resting blood pressure.
- This suggests a combined effect of age and blood pressure on risk.

**Resting Blood Pressure vs Maximum Heart Rate**
- Patients with higher resting blood pressure tend to achieve lower maximum heart rates.
- This may indicate cardiovascular limitation, particularly in patients with heart disease.

**Oldpeak vs Maximum Heart Rate**
- Higher exercise-induced ST depression is associated with lower maximum heart rates.
- This relationship is especially evident in the heart disease group.

**Oldpeak vs Age**
- Older patients tend to show slightly higher oldpeak values, more visible among patients with heart disease.

**Cholesterol vs Other Features**
- Cholesterol shows weak relationships with age, resting blood pressure, and maximum heart rate.
- This reinforces that cholesterol alone is not a strong distinguishing feature.

### Pairplot insights / feature interactions

- Combining age with maximum heart rate or oldpeak provides better separation of heart disease patients than any single feature alone.
- Visual clustering suggests interactions between features may be useful for feature engineering.
- Overall, age, maximum heart rate, resting blood pressure, and oldpeak are the most informative numeric features; cholesterol is less informative on its own.

In [ ]:
sns.pairplot(df[['age','chol','trestbps','thalach','oldpeak','target']], hue='target', palette='Set1')
plt.show()

### Numeric data analysis summary

#### What we did

- Performed **Exploratory Data Analysis (EDA)** on numeric features: age, cholesterol, resting blood pressure, maximum heart rate, and exercise-induced ST depression.
- Computed **summary statistics**: mean, median, quartiles, min, max, range, and standard deviation.
- Compared numeric features **by target** (heart disease vs. no heart disease).
- Visualized distributions using boxplots, violin plots, histograms, boxen plots, and pairplots.
- Interpreted differences, density patterns, outliers, and potential correlations to assess associations with heart disease.

#### Key findings

- **Age**: patients with heart disease tend to be older; distributions peak around 55–65 years.

- **Cholesterol**: slightly higher in patients with heart disease; distributions largely overlap, making it a weak standalone predictor.

- **Resting blood pressure**: slightly higher in patients with heart disease; moderately associated with risk.

- **Maximum heart rate**: lower in patients with heart disease; strongly associated with risk.

- **Exercise-induced ST depression**: higher values are more common in patients with heart disease; shows clear separation between groups, indicating strong predictive value.

- **Feature interactions**: negative correlation between age and maximum heart rate; slight positive correlation between age and resting blood pressure; negative relationship between oldpeak and maximum heart rate. Combining features improves differentiation between patient groups.

#### Summary

- Patients with heart disease are generally **older**, exhibit **higher resting blood pressure**, **higher oldpeak**, and **lower cardiovascular capacity** (maximum heart rate).
- **Age, maximum heart rate, and oldpeak** are the most informative numeric features.
- **Cholesterol** is less informative on its own.
- Analyzing distributions, quartiles, and outliers provides deeper insight than summary statistics alone, reinforcing the value of comprehensive numeric EDA.

-------------------------------------------------------------

### Performing categorical EDA

In this section we perform EDA on categorical features.

Goals:

- Compute summary statistics for categorical variables (frequency counts, relative frequency, distribution vs target).
- Compare categorical features by target (heart disease vs. no heart disease):
  - Sex: biological sex (0 = female, 1 = male)
  - Chest pain type (`cp`): (1 = typical, 2 = atypical, 3 = non-anginal, 4 = asymptomatic)
  - Exercise-induced angina (`exang`): (0 = no, 1 = yes)
  - Fasting blood sugar > 120 mg/dl (`fbs`): (0 = false, 1 = true)
  - Resting ECG result (`restecg`): (0 = normal, 1 = ST-T abnormality, 2 = LVH)
  - ST segment slope (`slope`): (1 = upsloping, 2 = flat, 3 = downsloping)
  - Number of major vessels (`ca`): (0–3)
  - Thalassemia status (`thal`): (3 = normal, 6 = fixed defect, 7 = reversible defect)
- Use graphs to identify correlations.
- Interpret the results.

-------------------------------------------------------------

### Sex — statistical summary

This table summarizes the `sex` feature (0 = female, 1 = male).

| Statistic | Female | Male |
|---|---:|---:|
| Count | 96 | 201 |
| Percent (approx.) | 32% | 68% |

There are over twice as many males as females in this sample.

| Target by count | Female | Male |
|---|---:|---:|
| No heart disease (0) | 71 | 89 |
| Has heart disease (1) | 25 | 112 |

| Target by percent (approx.) | Female | Male |
|---|---:|---:|
| No heart disease (0) | 74% | 44% |
| Has heart disease (1) | 26% | 56% |

Observed: males have a higher observed prevalence of heart disease in this sample (~56% of males vs ~26% of females). This indicates `sex` is associated with different observed rates in this dataset.

In [ ]:
df['sex'].value_counts()

In [ ]:
df['sex'].value_counts(normalize='index').round(2) * 100

In [ ]:
pd.crosstab(df['sex'], df['target'])

In [ ]:
pd.crosstab(df['sex'], df['target'], normalize='index').round(2) * 100

### Chest pain type (`cp`) — statistical summary

Chest pain type key: (1 = typical, 2 = atypical, 3 = non-anginal, 4 = asymptomatic)

| Statistic | Typical | Atypical | Non-anginal | Asymptomatic |
|---|---:|---:|---:|---:|
| Count | 23 | 49 | 83 | 142 |
| Percent (approx.) | 8% | 16% | 28% | 48% |

The dataset contains a plurality of asymptomatic cases (48%).

| Target by count | Typical | Atypical | Non-anginal | Asymptomatic |
|---|---:|---:|---:|---:|
| No heart disease (0) | 16 | 40 | 65 | 39 |
| Has heart disease (1) | 7 | 9 | 18 | 103 |

| Target by percent (approx.) | Typical | Atypical | Non-anginal | Asymptomatic |
|---|---:|---:|---:|---:|
| No heart disease (0) | 70% | 82% | 78% | 27% |
| Has heart disease (1) | 30% | 18% | 22% | 73% |

Interpretation: asymptomatic patients account for a large share of those with heart disease in this sample (73%). In this dataset, being asymptomatic is associated with a higher observed proportion of heart disease; this may reflect dataset characteristics or clinical coding and should be interpreted carefully.

In [ ]:
df['cp'].value_counts()

In [ ]:
df['cp'].value_counts(normalize='index').round(2) * 100

In [ ]:
pd.crosstab(df['cp'], df['target'])

In [ ]:
pd.crosstab(df['cp'], df['target'], normalize='index').round(2) * 100

### Exercise-induced angina (`exang`) — statistical summary

Exercise-induced angina key: (0 = no, 1 = yes)

| Statistic | No | Yes |
|---|---:|---:|
| Count | 200 | 97 |
| Percent (approx.) | 67% | 33% |

There are roughly twice as many people reporting no exercise-induced angina as those reporting yes.

| Target by count | No | Yes |
|---|---:|---:|
| No heart disease (0) | 137 | 23 |
| Has heart disease (1) | 63 | 74 |

| Target by percent (approx.) | No | Yes |
|---|---:|---:|
| No heart disease (0) | 68% | 24% |
| Has heart disease (1) | 32% | 76% |

Interpretation: about 76% of patients who reported exercise-induced angina had heart disease, versus ~32% among those who did not. This suggests exercise-induced angina is strongly associated with heart disease in this sample.

In [ ]:
df['exang'].value_counts()

In [ ]:
df['exang'].value_counts(normalize='index').round(2) * 100

In [ ]:
pd.crosstab(df['exang'], df['target'])

In [ ]:
pd.crosstab(df['exang'], df['target'], normalize='index').round(2) * 100

### Fasting Blood Sugar > 120 mg/dl (`fbs`) — statistical summary

This table summarizes the `fbs` feature (0 = false, 1 = true).

Fasting Blood Sugar > 120 mg/dl key: (0 = false, 1 = true)

| Statistics | False | True |
|---|---:|---:|
| By number of people in the study | 254 | 43 |
| By percent of people in the study (approx.) | 86% | 14% |

About 86% of participants have `fbs` = 0 and about 14% have `fbs` = 1.

| Target variable by count | False | True |
|---|---:|---:|
| No Heart Disease (0) | 137 | 23 |
| Has Heart Disease (1) | 117 | 20 |

| Target variable by percent (approx.) | False | True |
|---|---:|---:|
| No Heart Disease (0) (rounded) | 54% | 53% |
| Has Heart Disease (1) (rounded) | 46% | 47% |

The distribution is roughly balanced across the target; `fbs` alone appears to have little effect on heart disease prediction in this sample.

In [ ]:
df['fbs'].value_counts()

In [ ]:
df['fbs'].value_counts(normalize='index').round(2) * 100

In [ ]:
pd.crosstab(df['fbs'], df['target'])

In [ ]:
pd.crosstab(df['fbs'], df['target'], normalize='index').round(2) * 100

### Resting ECG (`restecg`) — statistical summary

This table summarizes the `restecg` feature (0 = normal, 1 = ST-T abnormality, 2 = left ventricular hypertrophy).

| Statistic | Normal | ST-T Abnormality | LVH |
|---|---:|---:|---:|
| Count | 147 | 4 | 146 |
| Percent (approx.) | 49% | 1% | 49% |

Normal and LVH categories are roughly evenly split; ST-T abnormality is rare (~1%).

| Target by count | Normal | ST-T Abnormality | LVH |
|---|---:|---:|---:|
| No heart disease (0) | 92 | 1 | 67 |
| Has heart disease (1) | 55 | 3 | 79 |

| Target by percent (approx.) | Normal | ST-T Abnormality | LVH |
|---|---:|---:|---:|
| No heart disease (0) | 63% | 25% | 46% |
| Has heart disease (1) | 37% | 75% | 54% |

Interpretation: fewer than 50% of patients with a normal resting ECG have heart disease, whereas a majority with LVH have heart disease. There are too few ST-T abnormality cases to draw reliable conclusions.

In [ ]:
df['restecg'].value_counts()

In [ ]:
df['restecg'].value_counts(normalize='index').round(2) * 100

In [ ]:
pd.crosstab(df['restecg'], df['target'])

In [ ]:
pd.crosstab(df['restecg'], df['target'], normalize='index').round(2) * 100

### ST segment slope (`slope`) — statistical summary

ST segment slope key: (1 = upsloping, 2 = flat, 3 = downsloping)

| Statistic | Up-sloping | Flat | Down-sloping |
|---|---:|---:|---:|
| Count | 139 | 137 | 21 |
| Percent (approx.) | 47% | 46% | 7% |

Up-sloping and flat slopes are common; down-sloping is less common (~7%).

| Target by count | Up-sloping | Flat | Down-sloping |
|---|---:|---:|---:|
| No heart disease (0) | 103 | 48 | 9 |
| Has heart disease (1) | 36 | 89 | 12 |

| Target by percent (approx.) | Up-sloping | Flat | Down-sloping |
|---|---:|---:|---:|
| No heart disease (0) | 74% | 35% | 43% |
| Has heart disease (1) | 26% | 65% | 57% |

Interpretation: flat and down-sloping ST segments show higher observed heart disease prevalence compared with up-sloping segments.

In [ ]:
df['slope'].value_counts()

In [ ]:
df['slope'].value_counts(normalize='index').round(2) * 100

In [ ]:
pd.crosstab(df['slope'], df['target'])

In [ ]:
pd.crosstab(df['slope'], df['target'], normalize='index').round(2) * 100

### Number of major vessels (`ca`) — statistical summary

Number of major vessels colored by fluoroscopy (0–3).

| Statistic | 0 | 1 | 2 | 3 |
|---|---:|---:|---:|---:|
| Count | 174 | 65 | 38 | 20 |
| Percent (approx.) | 59% | 22% | 13% | 7% |

About 60% of patients have 0 vessels colored; counts decline as the number of vessels increases.

| Target by count | 0 | 1 | 2 | 3 |
|---|---:|---:|---:|---:|
| No heart disease (0) | 129 | 21 | 7 | 3 |
| Has heart disease (1) | 45 | 44 | 31 | 17 |

| Target by percent (approx.) | 0 | 1 | 2 | 3 |
|---|---:|---:|---:|---:|
| No heart disease (0) | 74% | 32% | 18% | 15% |
| Has heart disease (1) | 26% | 68% | 82% | 85% |

Interpretation: a higher number of vessels colored by fluoroscopy is strongly associated with heart disease risk.

In [ ]:
df['ca'].value_counts()

In [ ]:
df['ca'].value_counts(normalize='index').round(2) * 100

In [ ]:
pd.crosstab(df['ca'], df['target'])

In [ ]:
pd.crosstab(df['ca'], df['target'], normalize='index').round(2) * 100

### Thalassemia status (`thal`) — statistical summary

Thalassemia key: (3 = normal, 6 = fixed defect, 7 = reversible defect)

| Statistic | Normal | Fixed defect | Reversible defect |
|---|---:|---:|---:|
| Count | 164 | 18 | 115 |
| Percent (approx.) | 55% | 6% | 39% |

Most patients are normal or have a reversible defect; a smaller group have a fixed defect.

| Target by count | Normal | Fixed defect | Reversible defect |
|---|---:|---:|---:|
| No heart disease (0) | 127 | 6 | 27 |
| Has heart disease (1) | 37 | 12 | 88 |

| Target by percent (approx.) | Normal | Fixed defect | Reversible defect |
|---|---:|---:|---:|
| No heart disease (0) | 77% | 33% | 23% |
| Has heart disease (1) | 23% | 67% | 77% |

Interpretation: patients with reversible or fixed thalassemia defects show substantially higher observed heart disease prevalence; reversible defects appear particularly associated with higher observed risk.

In [ ]:
df['thal'].value_counts()

In [ ]:
df['thal'].value_counts(normalize='index').round(2) * 100

In [ ]:
pd.crosstab(df['thal'], df['target'])

In [ ]:
pd.crosstab(df['thal'], df['target'], normalize='index').round(2) * 100

### Categorical summary

Most categorical features show moderate to strong associations with heart disease risk, except fasting blood sugar, which appears less informative in this dataset.

### Interpreting categorical data through graphs

I will provide countplots for categorical features to visualize category counts by target. These visuals complement the crosstabs shown earlier and make patterns easier to interpret.

### Countplot — Sex

This countplot shows males have a noticeably higher observed prevalence of heart disease than females in this sample.

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='sex', hue='target', data=df)
plt.title("Sex Distribution by Heart Disease Status")
plt.xlabel("Sex (0=Female, 1=Male)")
plt.ylabel("Count")
plt.legend(title="Heart Disease (0=No, 1=Yes)")
plt.show()

### Countplot — Chest pain type (`cp`)

This chart shows that patients with asymptomatic chest pain have a much higher observed proportion of heart disease compared to other chest pain types.

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='cp', hue='target', data=df)
plt.title("Chest Pain Type by Heart Disease Status")
plt.xlabel("Chest Pain Type")
plt.ylabel("Count")
plt.legend(title="Heart Disease (0=No, 1=Yes)")
plt.show()

### Countplot — Exercise-induced angina (`exang`)

Having exercise-induced angina is strongly associated with higher observed heart disease risk in this dataset.

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='exang', hue='target', data=df)
plt.title("Exercise-Induced Angina by Heart Disease Status")
plt.xlabel("Exercise-Induced Angina (0=No, 1=Yes)")
plt.ylabel("Count")
plt.legend(title="Heart Disease (0=No, 1=Yes)")
plt.show()

### Countplot — Fasting blood sugar (`fbs`)

The distribution of fasting blood sugar (<=120 mg/dL vs >120 mg/dL) is similar across heart disease status in this sample, suggesting `fbs` is not a strong predictor here.

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='fbs', hue='target', data=df)
plt.title("Fasting Blood Sugar by Heart Disease Status")
plt.xlabel("Fasting Blood Sugar (0=No, 1=Yes)")
plt.ylabel("Count")
plt.legend(title="Heart Disease (0=No, 1=Yes)")
plt.show()

### Countplot — Resting ECG (`restecg`)

The plot shows patients with left ventricular hypertrophy (LVH) have higher observed heart disease prevalence than those with a normal resting ECG.

There are too few ST-T abnormality cases to draw reliable predictive conclusions, though the limited data suggest higher observed risk.

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='restecg', hue='target', data=df)
plt.title("Resting ECG by Heart Disease Status")
plt.xlabel("Resting ECG (0=Normal, 1=ST-T wave abnormality, 2=Left ventricular hypertrophy)")
plt.ylabel("Count")
plt.legend(title="Heart Disease (0=No, 1=Yes)")
plt.show()

### Countplot — ST segment slope (`slope`)

Flat and downsloping ST segments show higher observed heart disease prevalence compared with upsloping segments.

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='slope', hue='target', data=df)
plt.title("Slope of ST Segment by Heart Disease Status")
plt.xlabel("Slope of ST Segment (0=Upsloping, 1=Flat, 2=Downsloping)")
plt.ylabel("Count")
plt.legend(title="Heart Disease (0=No, 1=Yes)")
plt.show()

### Countplot — Number of Major Vessels (`ca`)

There is a steady increase in observed heart disease prevalence as the number of major vessels colored by fluoroscopy increases.

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='ca', hue='target', data=df)
plt.title("Number of Major Vessels by Heart Disease Status")
plt.xlabel("Number of Major Vessels (0-3)")
plt.ylabel("Count")
plt.legend(title="Heart Disease (0=No, 1=Yes)")
plt.show()

### Countplot — Thalassemia status (`thal`)

Patients with a **fixed defect** or **reversible defect** show a substantially higher observed prevalence of heart disease compared with those with normal thalassemia status.

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='thal', hue='target', data=df)
plt.title("Thalassemia by Heart Disease Status")
plt.xlabel("Thalassemia (0=Normal, 1=Fixed Defect, 2=Reversible Defect)")
plt.ylabel("Count")
plt.legend(title="Heart Disease (0=No, 1=Yes)")
plt.show()

### Countplot Summary

In summary, the plots largely confirm patterns observed in the crosstabs. However, the visuals make these patterns easier to interpret and communicate.

### Categorical Data Analysis Summary

#### What we did

- Conducted statistical analysis of categorical features:
  - Counts and percentages for each category.
  - Distribution of the target variable by category (counts and percentages).

#### Key findings

- **Sex**
  - About twice as many males as females in the sample.
  - Observed prevalence of heart disease is higher in males (~56%) than females (~26%).

- **Chest pain type**
  - Most cases are asymptomatic (48%).
  - Asymptomatic patients account for a large share of heart disease cases (~73%); this may reflect dataset characteristics or coding and should be interpreted cautiously.

- **Exercise-induced angina**
  - Roughly twice as many patients reported no exercise-induced angina as those who did.
  - About 76% of patients with exercise-induced angina have heart disease, versus ~32% without — a strong association.

- **Fasting blood sugar (>120 mg/dl)**
  - Most participants (~86%) have `fbs` = 0; ~14% have `fbs` = 1.
  - The distribution across the target is similar, suggesting `fbs` is not strongly predictive in this sample.

- **Resting ECG**
  - Normal and LVH categories are roughly evenly split; ST-T abnormality is rare (~1%).
  - LVH shows higher observed heart disease prevalence.

- **ST segment slope**
  - Upsloping and flat slopes are common; downsloping is uncommon (~7%).
  - Flat and downsloping segments show higher observed heart disease prevalence compared with upsloping segments.

- **Number of major vessels colored by fluoroscopy**
  - Most patients (~60%) have 0 vessels colored; prevalence increases with the number of vessels.
  - A higher number of colored vessels is strongly associated with heart disease.

- **Thalassemia status**
  - Most patients are normal or have a reversible defect; fewer have fixed defects.
  - Reversible and fixed defects are associated with substantially higher observed heart disease prevalence.

#### Summary

- Most categorical features show some association with heart disease risk; `fbs` is the notable exception.
- Crosstabs and visualizations together provide a consistent picture of which categorical features merit further modeling.

## Conduct Feature Engineering and Risk Stratification

**Project Goal:**

In this section I add clinical structure to the dataset by creating meaningful groupings and derived features based on patterns observed during Exploratory Data Analysis. The goal is to prepare a final set of features suitable for modeling while preserving interpretability.

**Objectives:**

- Planned steps:
  - Create clinically meaningful **risk stratification** for key variables.
    - Example: split `chol` into two categories:
      - `chol >= 200 mg/dL` → 1
      - `chol < 200 mg/dL` → 0
  - Collapse or simplify less informative subcategories:
    - Drop the rare `restecg` ST-T abnormality category if needed.
    - Collapse chest pain type (`cp`) into **asymptomatic** (1) vs. other (0).

- Not doing in this section:
  - Variable transformation or nonlinear expansions.
  - Scaling or standardization.
  - Complex feature engineering that would reduce interpretability.

### Categorizing Cholesterol

We will transform `chol` from a numerical value to a categorical indicator. During exploratory analysis, the continuous distribution of cholesterol values did not show clear separation between patients with and without heart disease. To improve interpretability, cholesterol is stratified into a binary risk indicator.

Cholesterol after feature engineering key:

- `Cholesterol >= 200 mg/dL`: 1
- `Cholesterol < 200 mg/dL`: 0

| Statistics | Cholesterol < 200 mg/dL (0) | Cholesterol ≥ 200 mg/dL (1) |
|---|---:|---:|
| By number of people in the study | 48 | 249 |
| By percent of people in the study (rounded) | 16% | 84% |

There are about five times as many people with `chol >= 200 mg/dL` as those with `chol < 200 mg/dL`.

| Target Variable by Number | Cholesterol < 200 mg/dL (0) | Cholesterol ≥ 200 mg/dL (1) |
|---|---:|---:|
| No Heart Disease (0) | 28 | 132 |
| Has Heart Disease (1) | 20 | 117 |

| Target Variable by Percentage | Cholesterol < 200 mg/dL (0) | Cholesterol ≥ 200 mg/dL (1) |
|---|---:|---:|
| No Heart Disease (0) (rounded) | 58% | 53% |
| Has Heart Disease (1) (rounded) | 42% | 47% |

We observe only a small difference between the two categories, though the higher-cholesterol group shows a slightly higher prevalence of heart disease.

In [ ]:
df["chol"] = (df["chol"] >= 200).astype(int)

In [ ]:
print(df["chol"].value_counts())
print(df["chol"].value_counts(normalize='index').round(2) * 100)
print(pd.crosstab(df["chol"], df["target"]))
print(pd.crosstab(df["chol"], df["target"], normalize='index').round(2) * 100)

### Collapsing Chest Pain Type

Chest pain type (cp) is a multi-category variable describing different pain presentations. Among these, asymptomatic chest pain is clinically distinct and often associated with higher cardiovascular risk.

Chest pain type after feature engineering key:

- All other types = 0
- Asymptomatic chest pain = 1

| Statistics | All other types (0) | Asymptomatic (1) |
|--------|--------------|-------|
| By # of People in the Study | 154 | 139 |
| By % of People in the Study (rounded) | 53% | 47% |

We can see our feature is now pretty evenly distributed.

| Target Variable by Number | All other types (0) | Asymptomatic (1) |
|--------|--------------|-------|
| No Heart Disease (0) | 120 | 34 |
| Has Heart Disease (1) | 34 | 100 |

#

| Target Variable by Percentage | All other types (0) | Asymptomatic (1) |
|--------|--------------|-------|
| No Heart Disease (0) (rounded)| 78% | 28% |
| Has Heart Disease (1) (rounded)| 22% | 72% |

This did not change the percentages substantially. The feature is now much simpler and more evenly distributed.

In [ ]:
df["cp"] = (df["cp"] == 4).astype(int)

In [ ]:
print(df["cp"].value_counts())
print(df["cp"].value_counts(normalize='index').round(2) * 100)
print(pd.crosstab(df["cp"], df["target"]))
print(pd.crosstab(df["cp"], df["target"], normalize='index').round(2) * 100)

### Dropping ST–T Segment in Resting ECG (`restecg`)

The ST–T segment category in resting ECG (`restecg`) will be removed due to its low frequency. With only four samples, no reliable predictive conclusions can be drawn; removing this rare category prevents potential spurious correlations.

Resting ECG after feature engineering key:

- Normal = 0
- Left ventricular hypertrophy = 1

| Statistics | Normal (0) | Left Ventricular Hypertrophy (1) |
|---|---:|---:|
| By number of people in the study | 147 | 146 |
| By percent of people in the study (rounded) | 50% | 50% |

The split remains approximately even after removing the rare category.

Note: this operation drops four rows where `restecg` == 1.

In [ ]:
df = df[df["restecg"] != 1]
df['restecg'] = df['restecg'].replace({0:0, 2:1})

In [ ]:
print(df["restecg"].value_counts())
print(df["restecg"].value_counts(normalize='index').round(2) * 100)

### Summary of Feature Engineering and Risk Stratification

Feature engineering and risk stratification are now complete.

Summary of changes:

- `chol` was transformed to a binary risk factor:
  - `chol >= 200 mg/dL` → 1
  - `chol < 200 mg/dL` → 0
- Chest pain type (`cp`) was collapsed into a binary indicator:
  - Asymptomatic → 1
  - All other types → 0
- The rare ST–T wave abnormality category from `restecg` was removed; remaining categories were re-coded:
  - Left ventricular hypertrophy → 1
  - Normal → 0

A total of four rows were dropped when removing the rare `restecg` category.

All engineered features are retained for modeling; although `fbs` showed little association with heart disease, it is kept to preserve data integrity and potential value in multivariate models.

-------------------------------------------------------------------------

## Build an Interpretable Logistic Regression Model

**Project Goal:**

In this section I build an interpretable logistic regression model to predict the presence of heart disease using the engineered feature set. The goal is to establish a transparent baseline model and evaluate its predictive performance.

**Objectives:**

- Separate features (`X`) and the target variable (`y`)
- Perform a train/test split
- Justify feature selection choices
- Train a logistic regression model
- Evaluate model performance using standard classification metrics
- Assess model strengths and limitations

-------------------------------------------------------------

### Selecting the Final Feature Set and Defining X and y

After feature engineering, we choose the final feature set and define our model variables. We will then split the dataset into features (`X`) and target (`y`).

`X` will include the following features:
- `age`
- `sex`
- `cp`
- `trestbps`
- `chol`
- `fbs`
- `restecg`
- `thalach`
- `exang`
- `oldpeak`
- `slope`
- `ca`
- `thal`

`y` is the target variable:
- `target`

The features and target are now separated and ready for model training.

In [ ]:
features = [
   'age',
   'sex',
   'cp',
   'trestbps',
   'chol',
   'fbs',
   'restecg',
   'thalach',
   'exang',
   'oldpeak',
   'slope',
   'ca',
   'thal'
]

X = df[features]
y = df["target"]

In [ ]:
print(X.shape)
print(y.shape)

### Performing Train/Test Split

To evaluate accuracy, the data must be split into two sets:
- Training set: used to fit the model
- Test set: hold-out set used to evaluate the trained model's accuracy

I will hold out 25% for the test set. This provides sufficient data for training while keeping enough samples for a meaningful evaluation.

The random state is set to `2026` for reproducibility.

The shape of our features is shown below:

| X_train | X_test | y_train | y_test |
|---|---:|---:|---:|
| (219, 13) | (74, 13) | (219,) | (74,) |

Lastly, we stratify `y` so that the class distribution is maintained in train and test splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=2026,
    stratify=y,
)

In [ ]:
print("Original distribution:")
print(y.value_counts(normalize=True))

print("\nTraining distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

### Training the Logistic Regression Model

Now it's time to train the model.

I selected Logistic Regression for this project. The model outputs probabilities in the range [0, 1]; we use a 0.5 threshold to convert probabilities into class labels (>= 0.5 → heart disease, < 0.5 → no heart disease). Probabilities are useful for clinical interpretation and risk stratification.

The objective here is interpretability rather than extensive hyperparameter tuning, so I will use the standard `LogisticRegression` implementation (no regularization variants) and set `max_iter=1000` to ensure convergence.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

### Generating Predictions on the Test Set

Generate predictions for the test set. Predicted labels are stored in `y_pred`. Predicted probabilities (when needed) can be obtained with `model.predict_proba(X_test)` and are useful for thresholding or calibration.

In [ ]:
y_pred = model.predict(X_test)


### Testing Model Performance

We now evaluate model performance using standard classification metrics:

- Precision
- Recall
- F1-score
- Support

| Type | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| No Heart Disease (0) | 0.88 | 0.93 | 0.90 | 40 |
| Has Heart Disease (1) | 0.91 | 0.85 | 0.88 | 34 |

In [ ]:
print(classification_report(y_test, y_pred))

### Model Performance Summary

The logistic regression model demonstrated strong and balanced performance in distinguishing between patients with and without heart disease:

- For patients without heart disease, the model achieved high recall (0.93), indicating that most healthy individuals were correctly identified.
- For patients with heart disease, the model achieved high precision (0.91), suggesting that when the model predicts disease, it is usually correct.
- The F1-scores for both classes (0.90 for no disease and 0.88 for disease) indicate a balanced tradeoff between precision and recall.
- Support values show the performance was evaluated on a reasonably balanced test set, reducing concerns about class imbalance.

Overall, these results suggest the model is a reliable, interpretable baseline classifier for heart disease risk while maintaining clinically meaningful tradeoffs between sensitivity and specificity.

### Showing Model Coefficients

After training the logistic regression model, the learned coefficients were extracted and organized to examine each feature's relative contribution to the model’s predictions. Each coefficient represents the association between a given feature and the log-odds of heart disease, holding other variables constant.

The extracted coefficients are shown below:

- cp: 1.310
- ca: 1.082
- sex: 0.668
- exang: 0.534
- slope: 0.510
- thal: 0.411
- restecg: 0.277
- oldpeak: 0.153
- trestbps: 0.008
- age: -0.005
- thalach: -0.014
- fbs: -0.030
- chol: -0.182

Positive coefficients indicate an increase in the model-predicted log-odds of heart disease as the feature value increases, while negative coefficients indicate a decrease. Clinical interpretation of these associations follows in the next section.

In [ ]:
coefficients = pd.Series(
    model.coef_[0],
    index=features
).sort_values()

coefficients

## Interpret Model Results And Provide Clinical Insights

**Project Goal:**

In this section I will interpret the results of the logistic regression model to understand how individual features contribute to heart disease risk. The goal is to translate model outputs into clinically meaningful insights without making causal claims.

**Objectives:**

- Examine the direction and magnitude of model coefficients
- Identify the strongest predictors of heart disease
- Interpret results in plain, clinically relevant language
- Compare findings with known cardiovascular risk factors
- Clearly distinguish association from causation

-------------------------------------------------------------

### Logistic Regression Coefficients

In logistic regression, model coefficients represent the association between each predictor and the log-odds of heart disease. A positive coefficient indicates higher odds of disease, while a negative coefficient indicates lower odds.

5.1 Summary of Logistic Regression Coefficients

After fitting the logistic regression model, the learned coefficients were examined to assess the relative contribution of each feature to the prediction of heart disease. Each coefficient represents the association between a feature and the log-odds of heart disease, holding all other variables constant.

Strongest Positive Associations:

- Chest pain type (cp: 1.310)
- Number of major vessels colored by fluoroscopy (ca: 1.082)

These variables showed the strongest associations with increased predicted heart disease risk. In particular, the elevated coefficient for chest pain type suggests that patients classified into higher-risk chest pain categories, especially asymptomatic presentations, were more likely to be predicted as having heart disease by the model.

Moderate Positive Associations:

- Sex (sex: 0.668)
- Exercise-induced angina (exang: 0.534)
- Slope of the ST segment (slope: 0.510)
- Thalassemia classification (thal: 0.411)
- Resting ECG results (restecg: 0.277)

These findings suggest that male sex, exercise-related angina symptoms, abnormal ST segment responses, and certain thalassemia or ECG patterns were associated with higher predicted heart disease risk. While their coefficients were smaller than those of chest pain and vessel count, they still contributed meaningfully to the model’s predictions.

Weak or Minimal Associations:

- Resting blood pressure (trestbps: 0.008)
- Age (age: −0.005)
- ST depression (oldpeak: 0.153)

These features contributed minimally to the model’s predictions when controlling for other variables. The near-zero coefficient for age suggests that, within this dataset and feature set, age alone provided limited additional predictive value beyond other clinical indicators.

Negative Associations:

- Maximum heart rate achieved (thalach: −0.014)
- Fasting blood sugar (fbs: −0.030)
- Cholesterol (chol: −0.182)

Higher values of these variables were associated with lower predicted log-odds of heart disease in the model. In particular, the negative coefficient for maximum heart rate achieved suggests that better exercise capacity may be associated with lower risk, which is consistent with clinical intuition. The negative cholesterol coefficient may reflect dataset-specific patterns, confounding, or the influence of other correlated cardiovascular variables rather than a true protective effect.

### Converting Coefficients to Odds Ratios

We convert model coefficients to odds ratios for easier clinical interpretation.

Odds ratio key:

- Odds ratio > 1 = increased odds of disease
- Odds ratio < 1 = decreased odds
- Odds ratio = 1 = no association

The table below shows the odds ratio for each feature and a brief interpretation:

| Feature | Odds Ratio | Interpretation |
|---|---:|---|
| cp | 3.71 | Patients with asymptomatic chest pain had ~3.7× higher odds of heart disease compared to other chest pain types. |
| ca | 2.95 | Each additional major vessel visualized by fluoroscopy was associated with nearly 3× higher odds of heart disease. |
| sex | 1.95 | Male patients had nearly double the odds of heart disease compared to female patients. |
| exang | 1.71 | Presence of exercise-induced angina was associated with ~70% higher odds of heart disease. |
| slope | 1.67 | Abnormal ST segment slope during exercise testing was associated with increased odds of heart disease. |
| thal | 1.51 | Certain thalassemia patterns were associated with ~50% higher odds of heart disease. |
| restecg | 1.32 | Abnormal resting ECG findings were associated with modestly increased odds of heart disease. |
| oldpeak | 1.17 | Greater ST depression during exercise was associated with a small increase in odds. |
| trestbps | 1.01 | Resting blood pressure showed minimal association after adjusting for other features. |
| age | 0.995 | Age showed no meaningful independent association in this model. |
| thalach | 0.99 | Higher maximum heart rate achieved was associated with slightly lower odds of heart disease. |
| fbs | 0.97 | Fasting blood sugar showed little independent association. |
| chol | 0.83 | Higher cholesterol levels were associated with lower odds in this dataset, likely reflecting confounding or dataset-specific effects.

In [ ]:
odds_ratios = coefficients.apply(lambda x: np.exp(x))
odds_ratios

### Identify the Strongest Predictors

The model highlights the following strongest predictors of heart disease in this dataset:

1. Chest pain type (`cp`): asymptomatic presentations are associated with substantially higher odds (~3.7×) of heart disease.
2. Number of major vessels colored by fluoroscopy (`ca`): each additional vessel greatly increases odds of disease (nearly 3× per vessel in this model).
3. Sex (`sex`): male sex is associated with roughly double the odds of heart disease compared with female sex.

These features consistently contribute most to the model’s predicted risk. Other variables such as exercise-induced angina (`exang`) and ST segment slope (`slope`) also show meaningful positive associations.

### Compare Findings with Known Cardiovascular Risk Factors

Overall, many results align with clinical expectations: chest pain characteristics, number of diseased vessels, sex, and exercise-induced angina are commonly associated with coronary disease in the literature.

Discrepancies to note:

- Age: widely reported as a strong risk factor, but it shows little independent predictive power here (likely due to correlations with other variables or dataset characteristics).
- Cholesterol: the model shows an inverse association in this dataset; this is unlikely to be causal and may reflect confounding, selection bias, or treatment effects in the sample.
- Resting blood pressure: despite known associations, it contributed minimally after adjusting for other features.

Possible explanations for these differences include selection bias, survivor bias, restricted variable ranges, measurement timing, and multivariable confounding. These limitations should be considered when interpreting model coefficients.

### Association vs Causation

This analysis identifies associations between features and observed heart disease status; it does not establish causation. Model coefficients and odds ratios reflect relationships in the sample and can be influenced by confounding, selection effects, measurement timing, and other biases.

Use these results to generate hypotheses and prioritize variables for further study, not to draw causal conclusions.

### Interpreted Model Summary

This section summarizes the model's key findings:

- Chest Pain Type (`cp`): Patients with asymptomatic chest pain show substantially higher odds (≈3.7×) of heart disease.
- Number of Major Vessels (`ca`): A greater number of vessels colored by fluoroscopy is strongly associated with higher risk.
- Biological Sex (`sex`): Male patients have a higher observed prevalence of heart disease compared with female patients.

Other features showed weaker associations in the multivariable model, which may reflect dataset characteristics or multivariable confounding.

--------------------

## Summarizing Key Findings and Next Steps

**Project Goal:**

In this section I will synthesize insights from the exploratory analysis and modeling results to summarize key findings, propose data-driven hypotheses, and outline future directions for analysis and validation.

**Objectives:**

- Summarize the most important EDA findings
- Highlight the most influential model predictors
- Discuss dataset and modeling limitations
- Propose next steps for future analysis and validation

-------------------------------------------------------------

### Project Level Summary

In this project, we analyzed the Cleveland UCI Heart Disease dataset to predict the risk of heart disease using interpretable machine learning.

We started by defining the target variable and creating a detailed variable map to document all features and their clinical meanings. The dataset was loaded, inspected, and cleaned, including replacing missing values and removing a few incomplete rows.

Next, we performed feature engineering and risk stratification, transforming certain variables into clinically meaningful categories while keeping all features that showed some level of correlation with heart disease.

Exploratory Data Analysis (EDA) was conducted to examine distributions, identify patterns, and explore associations between features and heart disease risk.

Finally, we trained an interpretable logistic regression model, evaluated its performance, and converted coefficients into odds ratios to provide clear, clinically meaningful insights.

### Key Exploratory Data Analysis Findings

During Exploratory Data Analysis, we examined distributions and relationships between features and heart disease status.

**Key Findings:**

- **Sex:** Men were over twice as likely to have heart disease compared to women (female prevalence ~26%).
- **Chest Pain Type (`cp`):** Many patients were asymptomatic (48%). Asymptomatic patients represented a large share of heart disease cases (~73%), suggesting a strong association.
- **Exercise-Induced Angina (`exang`):** 76% of patients reporting exercise-induced angina had heart disease, compared to 32% without — a strong correlation.
- **Fasting Blood Sugar (`fbs`):** Distribution showed little predictive effect on heart disease.
- **Resting ECG (`restecg`):** Left ventricular hypertrophy was associated with higher risk; ST-T abnormalities were too rare to analyze reliably.
- **ST Segment Slope:** Flat and downsloping segments were more common among patients with disease.
- **Number of Major Vessels (`ca`):** More vessels visualized correlated with higher heart disease risk.
- **Thalassemia (`thal`):** Reversible and fixed defects were associated with substantially higher observed risk.

### Hypotheses Formulation

Based on EDA and model results, we propose the following hypotheses:

1. **Chest Pain Type Hypothesis:** Patients with asymptomatic chest pain are more likely to have heart disease than those with other chest pain types.  

2. **Major Vessels Hypothesis:** Patients with a higher number of major vessels colored by fluoroscopy have increased risk of heart disease.  

3. **Exercise-Induced Angina Hypothesis:** Patients reporting exercise-induced angina are more likely to have heart disease compared to those who do not.

### Statistical Test of a Selected Hypothesis

**Selected Hypothesis:** Chest Pain Type and Heart Disease

- **Null Hypothesis (H0):** There is no association between chest pain type and the presence of heart disease.  
- **Alternative Hypothesis (H1):** Asymptomatic chest pain is associated with higher odds of heart disease.

**Test Used:** Chi-square test of independence between categorical variables cp (asymptomatic vs others) and target (heart disease presence).  

**Result:** The test shows a significant association (p < 0.01), indicating that asymptomatic chest pain is strongly associated with heart disease in this dataset.

**Interpretation:** Patients with asymptomatic chest pain have higher risk of heart disease, consistent with model findings and clinical reasoning.

### Summary of Model Results

The logistic regression model highlighted the most important predictors of heart disease:

**Strongest Predictors:**
- **Chest Pain Type (`cp`)**: Asymptomatic patients had ~3.7× higher odds of heart disease.
- **Number of Major Vessels (`ca`)**: Each additional vessel colored increased odds by ~3×.
- **Sex (`sex`)**: Male patients had nearly double the odds of heart disease compared with female patients.

**Moderate Predictors:**
- Exercise-induced angina, ST segment slope, thalassemia patterns, and resting ECG findings contributed meaningfully.

**Minimal or Negative Associations:**
- Age, fasting blood sugar, cholesterol, resting blood pressure, ST depression, and maximum heart rate showed little or negative association in this model, likely reflecting dataset-specific effects or correlations with stronger predictors.

### Limitations and Next Steps

**Limitations:**
- The dataset is relatively small (≈303 patients), which may limit generalizability.
- Some features (e.g., `fbs`, ST-T abnormalities) have low prevalence or limited predictive value.
- Negative coefficient directions (e.g., cholesterol) may reflect confounding or dataset-specific bias rather than true protective effects.
- Logistic regression assumes linearity in the log-odds and may not capture complex interactions.

**Future Directions:**
- Validate the model on larger, independent datasets.
- Explore additional clinical features (e.g., lifestyle factors, family history).
- Investigate nonlinear and interaction effects with more advanced models while preserving interpretability.
- Consider prospective studies to confirm the predictive value of key features such as chest pain type, number of major vessels, and exercise-induced angina.

# Conclusion

This project analyzed the Cleveland UCI Heart Disease dataset to predict heart disease risk using interpretable machine learning.

**Key Takeaways:**

- **Strongest predictors:** chest pain type (asymptomatic highest risk), number of major vessels colored by fluoroscopy, and biological sex (male higher risk).
- **Exploratory analysis** revealed clear assosiations between most features and heart disease risk.
- **Logistic regression** provided a transparent baseline model that performed well at discriminating cases and controls.

**Limitations:**
- Small dataset (≈303 patients) may limit generalizability.
- Some features were underrepresented or had limited predictive value.
- Certain coefficient directions (e.g., cholesterol) may reflect dataset-specific effects rather than clinical causation.

**Next Steps:**
- Validate the model on larger, independent datasets.
- Incorporate additional clinical variables and longitudinal data when available.
- Explore interaction and nonlinear effects while maintaining interpretability.
- Use these findings to prioritize hypotheses for follow-up clinical research.

# Thank You!

Thank you for reviewing this analysis. This concludes the project on predicting heart disease risk using the Cleveland UCI dataset.